# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

Found existing installation: pydantic_core 2.50.1
Uninstalling pydantic_core-2.50.1:
  Successfully uninstalled pydantic_core-2.50.1
Found existing installation: pydantic 2.14.1
Uninstalling pydantic-2.14.1:
  Successfully uninstalled pydantic-2.14.1
Note: you may need to restart the kernel to use updated packages.
  Using cached pydantic-1.10.26-cp312-cp312-win_amd64.whl.metadata (156 kB)
Using cached pydantic-1.10.26-cp312-cp312-win_amd64.whl (1.9 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.9 requires pydantic>=2.0, but you have pydantic 1.10.26 which is incompatible.
kubernetes 37.0.1 requires pydantic>=2.13.5, but you have pydantic 1.10.26 which is incompatible.
pydantic-settings 2.16.0 requires pydantic>=2.7.0, but you have pydantic 1.10.26 which is incompatible.
spacy 3.8.16 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
thinc 8.3.13 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
weasel 1.0.0 requires pydantic>=2.0.0, but you have pydantic 1.10.26 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached chromadb-0.3.21-py3-none-any.whl.metadata (5.7 kB)
  Using cached hnswlib-0.8.0.tar.gz (36 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached clickhouse_connect-1.11.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached duckdb-1.5.6-cp312-cp312-win_amd64.whl.metadata (4.2 kB)
  Using cached fastapi-0.151.0-py3-none-any.whl.metadata (26 kB)
  Using cached backports_zstd-1.8.0-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
  Using cached lz4-4.4.5-cp312-cp312-win_amd64.whl.metadata (3.9 kB)
  Using cached starlette-1.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (1

  error: subprocess-exited-with-error
  
  × Building wheel for hnswlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [5 lines of output]
      running bdist_wheel
      running build
      running build_ext
      building 'hnswlib' extension
      error: Unable to find a compatible Visual Studio installation.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for hnswlib

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Failed to build installable wheels for some pyproject.toml based projects (hnswlib)


  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.9 requires pydantic>=2.0, but you have pydantic 1.10.26 which is incompatible.
contourpy 1.4.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
spacy 3.8.16 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
thinc 8.3.13 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip uninstall -y pydantic pydantic-core numpy pyarrow datasets
%pip install -U "numpy>=2" pyarrow datasets pydantic
%pip install -U faiss-cpu chromadb sentence-transformers transformers

Found existing installation: pydantic 1.10.26
Uninstalling pydantic-1.10.26:
  Successfully uninstalled pydantic-1.10.26
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pyarrow 26.0.0
Uninstalling pyarrow-26.0.0:
  Successfully uninstalled pyarrow-26.0.0
Found existing installation: datasets 5.1.0
Uninstalling datasets-5.1.0:
  Successfully uninstalled datasets-5.1.0
Note: you may need to restart the kernel to use updated packages.


  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pyarrow-26.0.0-cp312-cp312-win_amd64.whl.metadata (3.0 kB)
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached pydantic_core-2.50.1-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached pyarrow-26.0.0-cp312-cp312-win_amd64.whl (28.6 MB)
Using cached datasets-5.1.0-py3-none-any.whl (604 kB)
Using cached pydantic-2.14.1-py3-none-any.whl (488 kB)
Using cached pydantic_core-2.50.1-cp312-cp312-win_amd64.whl (2.2 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# %pip install -U tf-keras

In [4]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display

os.makedirs('cache', exist_ok=True)


c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🌟 Exercise 1 · Data loading and preparation

In [5]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))
display(pdf.head())
pdf_subset = pdf.head(1000)
pdf_subset[['id', 'title']].head()


,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


,id,title
0,0,A closer look at water-splitting's solar fuel ...
1,1,"An irresistible scent makes locusts swarm, stu..."
2,2,Artificial intelligence warning: AI will know ...
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how ...


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [6]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

faiss_train_examples = [example_create_fn(idx, title) for idx, title in enumerate(pdf_subset['title'])]
faiss_train_examples[:2]


In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


Batches: 100%|██████████| 32/32 [00:02<00:00, 10.80it/s]


(1000, 384)

## 🌟 Exercise 3 · FAISS indexing and search

In [8]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


1000

In [9]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    query_vector = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)
    sims, ids = index_content.search(query_vector, k)
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results

display(search_content('animal', pdf_to_index, k=5))


,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [10]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)
collection = chroma_client.create_collection(name=collection_name)
collection.add(
    ids=[str(i) for i in pdf_subset['id']],
    documents=pdf_subset['title'].tolist(),
    metadatas=[{'topic': t} for t in pdf_subset['topic']]
)
results = collection.query(query_texts=['animal'], n_results=5)
print(json.dumps(results, indent=2))


{
  "ids": [
    [
      "176",
      "975",
      "99",
      "928",
      "762"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla",
      "Researchers explore social behavior of animals toward emerging infectious diseases",
      "Ghostwire: Tokyo confirms dog petting",
      "Just Let This Lizard Be a Dinosaur",
      "'Secret' life of sharks: Study reveals their surprising social networks"
    ]
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents",
    "distances"
  ],
  "data": null,
  "metadatas": [
    [
      {
        "topic": "TECHNOLOGY"
      },
      {
        "topic": "HEALTH"
      },
      {
        "topic": "TECHNOLOGY"
      },
      {
        "topic": "SCIENCE"
      },
      {
        "topic": "SCIENCE"
      }
    ]
  ],
  "distances": [
    [
      1.216196060180664,
      1.2464325428009033,
      1.3118830919265747,
      1.3652255535125732,
      1.409005880355835
  

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [11]:
# %pip install -U "transformers<5"

In [12]:
# model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA
# # To-Do: create the text2text-generation pipeline
# pipe = pipeline('text2text-generation', model=model_id)

# question = "What's the latest news on space development?"
# context_docs = results['documents'][0][:3]
# context = ' '.join(context_docs)
# prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
# response = pipe(prompt)[0]['generated_text']
# print(response)

from transformers import AutoModelForSeq2SeqLM

model_id = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_id)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

question = "What's the latest news on space development?"
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

inputs = tokenizer(prompt, return_tensors='pt', truncation=True)
output_ids = seq2seq_model.generate(**inputs, max_new_tokens=50)
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(response)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1688.90it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Ghostwire: Tokyo confirms dog petting
